# ハミルトニアンシミュレーションの回路資源比較

Trotter–Suzuki、QSVT、multiproduct formula (MPF) を、同じ Pauli 和と誤差予算の下で比較します。小規模系では実際の statevector、大規模系では密行列を作らないゲート分解モデルを使います。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from hamiltonian_resources import (
    BenchmarkConfig, benchmark_scaling, build_hamiltonian_qsvt_circuit,
    build_multiproduct_circuit, build_trotter_circuit, compare_with_exact,
    estimate_qsvt_degree, transverse_field_ising,
)
pd.set_option("display.max_columns", None)

## 1. ハミルトニアン入力

例として $H=-J\sum_i Z_iZ_{i+1}-h\sum_iX_i$ を使います。Qiskit の Pauli label は右端が qubit 0 です。任意の Pauli 和は `PauliHamiltonian.from_terms` で渡せます。

In [ ]:
H = transverse_field_ising(3, coupling=1.0, field=0.7)
print(H.name, H.terms)
print(f"terms={H.term_count}, alpha=sum|h_j|={H.alpha:.3f}")

## 2. 実回路を構成する

下の関数はいずれも Hamiltonian の密行列を指数化しません。QSVTは時刻と許容誤差からcosine/sine位相を合成し、MPFは既定の`new` scheduleでsegmentごとのrobust OAAを構成します。

In [ ]:
time = 0.2
trotter_circuit = build_trotter_circuit(H, time, reps=2, order=2)
mpf_circuit = build_multiproduct_circuit(H, time, m=2, segments=1)
qsvt_circuit = build_hamiltonian_qsvt_circuit(H, time, epsilon=1e-2)

display(trotter_circuit.draw(output="mpl", fold=30))
print("MPF metadata:", mpf_circuit.metadata)
print("QSVT metadata:", qsvt_circuit.metadata)

## 3. 小規模系で厳密解と比較

MPF は branch=0 への postselection 後の fidelity と成功確率を別々に表示します。

In [ ]:
validation = []
for reps in (1, 2, 4, 8):
    row = compare_with_exact(H, time, method="trotter", reps=reps, trotter_order=2)
    row["segments"] = reps
    validation.append(row)
for reps in (1, 2, 4):
    row = compare_with_exact(H, time, method="multiproduct", reps=reps, mpf_m=2)
    row["segments"] = reps
    validation.append(row)
pd.DataFrame(validation)

## 4. benchmark dataを生成・可視化

このsectionは`benchmark_config.json`に対応するCSVがない、または設定digestが古い場合だけ両sweepを生成します。初回実行には時間がかかりますが、以後は保存されたCSVを再利用します。Trotter $p=1,2,4,6$、MPF $m=3,5,7$、QSVTの8曲線に同じstyle mappingを適用します。

data schema、target-error sweep、best-of-family summary、解析上の仮定は`docs/resource_scaling_benchmarks.md`を参照してください。

In [ ]:
from pathlib import Path
from hamiltonian_resources import (
    create_benchmark_figure, generate_and_save_benchmark,
    load_benchmark_config, load_benchmark_data,
)

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "benchmark_config.json").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("benchmark_config.json が見つかりません")
benchmark_config = load_benchmark_config(project_root / "benchmark_config.json")
benchmark_dir = benchmark_config.output_directory
data_paths = {
    "system-size": benchmark_dir / "system_size_scaling.csv",
    "target-error": benchmark_dir / "target_error_scaling.csv",
}

def data_is_current(path):
    if not path.exists():
        return False
    try:
        frame = load_benchmark_data(path)
    except (OSError, ValueError):
        return False
    return set(frame["config_digest"].astype(str)) == {benchmark_config.digest}

if not all(data_is_current(path) for path in data_paths.values()):
    print("benchmark dataを現在の設定で生成します")
    for sweep in data_paths:
        frame, csv_path, _ = generate_and_save_benchmark(benchmark_config, sweep)
        skipped = int((frame["status"] == "skipped").sum())
        print(f"{csv_path.name}: {len(frame)} rows, {skipped} skipped")

system_size_resources = load_benchmark_data(benchmark_dir / "system_size_scaling.csv")
target_error_resources = load_benchmark_data(benchmark_dir / "target_error_scaling.csv")
system_size_resources[["system_qubits", "method_label", "t_count", "cnot_count", "status"]].head(16)

In [ ]:
from io import BytesIO
from IPython.display import Image, display

figures = (
    create_benchmark_figure(system_size_resources, "t_count"),
    create_benchmark_figure(system_size_resources, "cnot_count"),
    create_benchmark_figure(target_error_resources, "t_count"),
    create_benchmark_figure(target_error_resources, "cnot_count"),
)
for figure in figures:
    image_buffer = BytesIO()
    figure.savefig(image_buffer, format="png", dpi=160, bbox_inches="tight")
    display(Image(data=image_buffer.getvalue()))

## 5. 小規模で解析モデルを transpile 値に校正

実回路の多重制御分解は急速に大きくなります。まず緩い誤差・2 qubit で試し、その後に範囲を広げてください。

In [ ]:
small_config = BenchmarkConfig(time=0.05, target_error=0.2, mpf_m=2)
compiled = benchmark_scaling(
    [2], transverse_field_ising, small_config, transpile_circuits=True
)
compiled[["algorithm", "num_qubits", "t_count", "cnot_count", "depth"]]

### 解釈上の注意

- QSVT の PREPARE/SELECT コストは Hamiltonian の入力モデルに強く依存します。ここでは一般の Pauli-LCU を公平に数えています。
- 8構成とも決定的動作あたりの比較です。MPF と QSVT の解析値は segment / 回路ごとの 3-step robust OAA を含み、`nominal_success_probability` は 1 です。
- Trotter次数1, 2はChilds交換子上界、次数4, 6はgroup数がwork cap内ならSchubert--Mendl交換子上界を使います。fallbackはCSVの`bound_method`と`bound_rigorous`で明示されます。QSVTの次数は厳密なJacobi–Anger打ち切りです。MPFのsegment数だけは`alpha_eff = min(alpha, W2^(1/3))`による交換子校正proxyであり、小規模では`compare_with_exact`で校正してください。
- 解析モデルの controlled QSVT は、`V`/`V^dagger` が block-encoding query を共有し projector 位相のみ選択する効率的コンパイルを仮定します。上の `transpile_circuits=True` の校正値は Qiskit の汎用 `.control()` 分解を数えるため、解析値よりかなり大きくなります。
- family summaryは各x値で保存済みrowをpost-processした結果で、評価対象外のTrotter次数やMPF term数を最適化したものではありません。QSVTの優位は$\alpha t$が大きく、かつ誤差要求が厳しい領域（$\log(1/\varepsilon)$ scaling）で現れます。